# 🎓 DuoSentimen - Analisis Sentimen Ulasan Duolingo
## Menggunakan Metode Naive Bayes dengan Pembobotan TF-IDF
**By Ahmad Saifulla**

---

### Kerangka Kerja:
1. Dataset Ulasan Duolingo → 2. Case Folding → 3. Cleaning → 4. Tokenizing → 5. Stopword Removal → 6. Stemming → 7. Pembobotan TF-IDF → 8. Pembagian Data 80:20 → 9. Model Naive Bayes → 10. Klasifikasi Sentimen (Positif, Negatif, Netral) → 11. Evaluasi Model (Accuracy, Precision, Recall, F1-Score)

## 1. Import Library

In [ ]:
# Library utama
import pandas as pd
import numpy as np
import re
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Visualisasi
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score
)

# NLP Indonesia
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Konfigurasi plot
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('✅ Semua library berhasil diimport!')

## 2. Load Dataset
Dataset berupa file CSV ulasan aplikasi Duolingo dari Google Play Store.

Format kolom: `Review ID, Username, Rating, Review Text, Date`

In [ ]:
# Ganti nama file sesuai dengan file CSV Anda
FILE_CSV = 'hasil_scrapper_ulasan_app_duolingo.csv'

df = pd.read_csv(FILE_CSV)
print(f'📊 Jumlah data: {len(df)} ulasan')
print(f'📋 Kolom: {list(df.columns)}')
print()
df.head(10)

In [ ]:
# Informasi dataset
print('=== INFO DATASET ===')
df.info()
print(f'\nShape: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()}')

In [ ]:
# Distribusi Rating
fig, ax = plt.subplots(figsize=(8, 5))
rating_counts = df['Rating'].value_counts().sort_index()
colors = ['#e17055', '#e17055', '#fdcb6e', '#00b894', '#00b894']
rating_counts.plot(kind='bar', color=colors, edgecolor='white', ax=ax)
ax.set_title('Distribusi Rating Ulasan Duolingo', fontsize=16, fontweight='bold')
ax.set_xlabel('Rating (Bintang)', fontsize=12)
ax.set_ylabel('Jumlah Ulasan', fontsize=12)
ax.set_xticklabels(rating_counts.index, rotation=0)
for i, v in enumerate(rating_counts.values):
    ax.text(i, v + 10, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Labeling (Berdasarkan Rating)
- ⭐ Rating 1-2 → **Negatif**
- ⭐ Rating 3 → **Netral**
- ⭐ Rating 4-5 → **Positif**

In [ ]:
def auto_label(rating):
    """Labeling otomatis berdasarkan rating bintang"""
    try:
        rating = int(float(rating))
    except (ValueError, TypeError):
        return 'netral'
    if rating <= 2:
        return 'negatif'
    elif rating == 3:
        return 'netral'
    else:
        return 'positif'

df['label'] = df['Rating'].apply(auto_label)

# Distribusi label
label_counts = df['label'].value_counts()
print('📊 Distribusi Label Sentimen:')
for label, count in label_counts.items():
    pct = count / len(df) * 100
    print(f'  {label:>8}: {count:>5} ({pct:.1f}%)')

In [ ]:
# Visualisasi distribusi label
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

color_map = {'positif': '#00b894', 'negatif': '#e17055', 'netral': '#fdcb6e'}
colors = [color_map.get(l, '#888') for l in label_counts.index]

# Pie chart
axes[0].pie(label_counts.values, labels=label_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Distribusi Sentimen (Pie Chart)', fontsize=14, fontweight='bold')

# Bar chart
label_counts.plot(kind='bar', color=colors, edgecolor='white', ax=axes[1])
axes[1].set_title('Distribusi Sentimen (Bar Chart)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Label Sentimen')
axes[1].set_ylabel('Jumlah')
axes[1].set_xticklabels(label_counts.index, rotation=0)
for i, v in enumerate(label_counts.values):
    axes[1].text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Preprocessing Teks
Pipeline preprocessing:
1. **Case Folding** - Mengubah ke huruf kecil
2. **Cleaning** - Menghapus URL, mention, hashtag, angka, emoji, tanda baca
3. **Tokenizing** - Memecah teks menjadi kata-kata
4. **Stopword Removal** - Menghapus kata umum tidak bermakna
5. **Stemming** - Mengubah kata ke bentuk dasar (Sastrawi)

### 4.1 Case Folding

In [ ]:
def case_folding(text):
    """Mengubah teks menjadi huruf kecil (lowercase)"""
    if not isinstance(text, str):
        return ''
    return text.lower()

# Contoh
contoh = 'Aplikasi DUOLINGO Sangat BAGUS!!'
print(f'Sebelum: {contoh}')
print(f'Sesudah: {case_folding(contoh)}')

### 4.2 Cleaning

In [ ]:
def cleaning(text):
    """Membersihkan teks dari elemen yang tidak diperlukan"""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\.\S+', '', text)  # Hapus URL
    text = re.sub(r'@\w+', '', text)               # Hapus mention
    text = re.sub(r'#\w+', '', text)               # Hapus hashtag
    text = re.sub(r'\d+', '', text)                # Hapus angka
    text = re.sub(r'[^\x00-\x7F]+', '', text)     # Hapus emoji
    text = re.sub(r'[^\w\s]', '', text)            # Hapus tanda baca
    text = re.sub(r'\s+', ' ', text).strip()       # Hapus spasi berlebih
    return text

# Contoh
contoh = 'Cek https://duolingo.com @duolingo #belajar 123 bagus!! 😍🔥'
print(f'Sebelum: {contoh}')
print(f'Sesudah: {cleaning(contoh)}')

### 4.3 Tokenizing

In [ ]:
def tokenizing(text):
    """Memecah teks menjadi daftar kata (token)"""
    if not isinstance(text, str):
        return []
    return text.split()

# Contoh
contoh = 'aplikasi duolingo sangat bagus'
print(f'Sebelum: {contoh}')
print(f'Sesudah: {tokenizing(contoh)}')

### 4.4 Stopword Removal

In [ ]:
# Load stopwords
stopwords_file = 'stopwords.txt'
if os.path.exists(stopwords_file):
    with open(stopwords_file, 'r', encoding='utf-8') as f:
        stop_words = set([line.strip() for line in f if line.strip()])
    print(f'✅ Stopwords dimuat: {len(stop_words)} kata')
else:
    from nltk.corpus import stopwords
    import nltk
    nltk.download('stopwords', quiet=True)
    stop_words = set(stopwords.words('indonesian'))
    print(f'✅ Stopwords NLTK dimuat: {len(stop_words)} kata')

def remove_stopwords(tokens):
    """Menghapus stopwords dari daftar token"""
    return [w for w in tokens if w not in stop_words]

# Contoh
contoh = ['aplikasi', 'ini', 'sangat', 'bagus', 'untuk', 'belajar']
print(f'Sebelum: {contoh}')
print(f'Sesudah: {remove_stopwords(contoh)}')

### 4.5 Stemming (Sastrawi)

In [ ]:
# Inisialisasi Sastrawi Stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()
print('✅ Sastrawi stemmer berhasil diinisialisasi')

def stemming(tokens):
    """Stemming kata ke bentuk dasar menggunakan Sastrawi"""
    return [stemmer.stem(w) for w in tokens]

# Contoh
contoh = ['berlari', 'memakan', 'pembelajaran', 'menggunakan']
print(f'Sebelum: {contoh}')
print(f'Sesudah: {stemming(contoh)}')

### 4.6 Full Preprocessing Pipeline

In [ ]:
# Load kamus normalisasi jika ada
norm_dict = {}
if os.path.exists('normalisasi.json'):
    with open('normalisasi.json', 'r', encoding='utf-8') as f:
        norm_dict = json.load(f)
    print(f'✅ Kamus normalisasi dimuat: {len(norm_dict)} kata')

def normalisasi(text):
    """Normalisasi kata slang ke bentuk baku"""
    words = text.split()
    return ' '.join([norm_dict.get(w, w) for w in words])

def preprocess(text):
    """Pipeline preprocessing lengkap"""
    text = case_folding(text)        # 1. Case Folding
    text = cleaning(text)            # 2. Cleaning
    text = normalisasi(text)         # 3. Normalisasi
    tokens = tokenizing(text)        # 4. Tokenizing
    tokens = remove_stopwords(tokens) # 5. Stopword Removal
    tokens = stemming(tokens)        # 6. Stemming
    return ' '.join(tokens)

# Test preprocessing
test_texts = [
    'Aplikasi ini SANGAT BAGUS untuk belajar bahasa!!',
    'Tidak berguna, sering error dan lambat sekali',
    'Biasa saja, lumayan lah buat belajar'
]

print('=== Contoh Hasil Preprocessing ===')
for t in test_texts:
    print(f'\nInput : {t}')
    print(f'Output: {preprocess(t)}')

In [ ]:
# Terapkan preprocessing pada seluruh dataset
print('⏳ Memulai preprocessing dataset...')
df['ulasan_bersih'] = df['Review Text'].astype(str).apply(preprocess)

# Hapus baris yang hasilnya kosong
df = df[df['ulasan_bersih'].str.strip() != ''].reset_index(drop=True)

print(f'✅ Preprocessing selesai! {len(df)} data berhasil diproses')
print()

# Tampilkan contoh before-after
sample = df[['Review Text', 'ulasan_bersih', 'label']].head(10)
sample.columns = ['Ulasan Asli', 'Ulasan Bersih', 'Label']
sample

## 5. Pembobotan TF-IDF
TF-IDF (Term Frequency - Inverse Document Frequency) digunakan untuk mengubah teks menjadi representasi numerik yang dapat diproses oleh algoritma Naive Bayes.

In [ ]:
# TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X = tfidf_vectorizer.fit_transform(df['ulasan_bersih'])
y = df['label']

print(f'📐 Shape TF-IDF Matrix: {X.shape}')
print(f'📚 Ukuran Vocabulary: {len(tfidf_vectorizer.vocabulary_)} kata')
print(f'🏷️  Jumlah Label: {len(y)}')

In [ ]:
# Top 15 kata dengan bobot TF-IDF tertinggi
feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_mean = X.mean(axis=0).A1
top_idx = tfidf_mean.argsort()[-15:][::-1]

fig, ax = plt.subplots(figsize=(12, 6))
top_words = [feature_names[i] for i in top_idx]
top_scores = [tfidf_mean[i] for i in top_idx]
ax.barh(range(len(top_words)), top_scores, color='#667eea', edgecolor='white')
ax.set_yticks(range(len(top_words)))
ax.set_yticklabels(top_words)
ax.invert_yaxis()
ax.set_title('Top 15 Kata dengan Bobot TF-IDF Tertinggi', fontsize=14, fontweight='bold')
ax.set_xlabel('Rata-rata TF-IDF Score')
plt.tight_layout()
plt.show()

## 6. Pembagian Data (80:20)
Dataset dibagi menjadi:
- **80%** Data Training (untuk melatih model)
- **20%** Data Testing (untuk menguji model)

In [ ]:
# Split data 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'📊 Total Data      : {X.shape[0]}')
print(f'📗 Data Training   : {X_train.shape[0]} ({X_train.shape[0]/X.shape[0]*100:.0f}%)')
print(f'📕 Data Testing    : {X_test.shape[0]} ({X_test.shape[0]/X.shape[0]*100:.0f}%)')
print()
print('Distribusi label Training:')
print(y_train.value_counts())
print()
print('Distribusi label Testing:')
print(y_test.value_counts())

## 7. Model Naive Bayes
Menggunakan **Multinomial Naive Bayes** dengan Laplace Smoothing (alpha=1).

In [ ]:
# Training model Naive Bayes
model = MultinomialNB(alpha=1.0)
model.fit(X_train, y_train)

print('✅ Model Naive Bayes berhasil di-training!')
print(f'📋 Kelas yang dipelajari: {list(model.classes_)}')

## 8. Klasifikasi Sentimen (Prediksi)

In [ ]:
# Prediksi pada data testing
y_pred = model.predict(X_test)

# Tampilkan contoh hasil prediksi
hasil_df = pd.DataFrame({
    'Ulasan': df.loc[y_test.index, 'Review Text'].values[:15],
    'Label Asli': y_test.values[:15],
    'Label Prediksi': y_pred[:15],
    'Status': ['✅ Benar' if a == p else '❌ Salah' for a, p in zip(y_test.values[:15], y_pred[:15])]
})

print('=== Contoh Hasil Klasifikasi ===')
hasil_df

## 9. Evaluasi Model
Metrik evaluasi: **Accuracy, Precision, Recall, F1-Score**

In [ ]:
# Hitung metrik evaluasi
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

print('=' * 50)
print('        HASIL EVALUASI MODEL')
print('=' * 50)
print(f'  Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'  Precision : {precision:.4f} ({precision*100:.2f}%)')
print(f'  Recall    : {recall:.4f} ({recall*100:.2f}%)')
print(f'  F1-Score  : {f1:.4f} ({f1*100:.2f}%)')
print('=' * 50)
print()
print('\n=== Classification Report ===')
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Visualisasi metrik evaluasi
fig, ax = plt.subplots(figsize=(10, 6))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [accuracy * 100, precision * 100, recall * 100, f1 * 100]
colors = ['#0984e3', '#00b894', '#e17055', '#6c5ce7']

bars = ax.bar(metrics, values, color=colors, edgecolor='white', width=0.6)
ax.set_ylim(0, 110)
ax.set_title('Metrik Evaluasi Model Naive Bayes + TF-IDF', fontsize=16, fontweight='bold')
ax.set_ylabel('Nilai (%)', fontsize=12)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{val:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=['positif', 'negatif', 'netral'])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Positif', 'Negatif', 'Netral'],
            yticklabels=['Positif', 'Negatif', 'Netral'],
            ax=ax, annot_kws={'size': 14})
ax.set_title('Confusion Matrix', fontsize=16, fontweight='bold')
ax.set_xlabel('Label Prediksi', fontsize=12)
ax.set_ylabel('Label Asli', fontsize=12)
plt.tight_layout()
plt.show()

## 10. Word Cloud
Visualisasi kata-kata yang paling sering muncul per kelas sentimen.

In [ ]:
# Word Cloud per kelas sentimen
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sentimen_config = [
    ('positif', 'Greens', '🟢 Positif'),
    ('negatif', 'Reds', '🔴 Negatif'),
    ('netral', 'YlOrBr', '🟡 Netral')
]

for ax, (label, cmap, title) in zip(axes, sentimen_config):
    text = ' '.join(df[df['label'] == label]['ulasan_bersih'].values)
    if text.strip():
        wc = WordCloud(width=600, height=400, background_color='white',
                      colormap=cmap, max_words=100, random_state=42).generate(text)
        ax.imshow(wc, interpolation='bilinear')
    else:
        ax.text(0.5, 0.5, 'Tidak ada data', ha='center', va='center', fontsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')

plt.suptitle('Word Cloud per Kelas Sentimen', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 11. Diagram Frekuensi Kata

In [ ]:
# Top 20 kata yang paling sering muncul
from collections import Counter

all_words = ' '.join(df['ulasan_bersih']).split()
word_freq = Counter(all_words).most_common(20)

words, counts = zip(*word_freq)

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(range(len(words)), counts, color='#667eea', edgecolor='white')
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words, fontsize=12)
ax.invert_yaxis()
ax.set_title('Top 20 Kata yang Paling Sering Muncul', fontsize=16, fontweight='bold')
ax.set_xlabel('Frekuensi', fontsize=12)

for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 12. Prediksi Sentimen Manual
Uji coba prediksi sentimen pada teks baru.

In [ ]:
def predict_sentiment(text):
    """Prediksi sentimen dari teks baru"""
    cleaned = preprocess(text)
    vector = tfidf_vectorizer.transform([cleaned])
    result = model.predict(vector)
    proba = model.predict_proba(vector)[0]
    return result[0], dict(zip(model.classes_, proba))

# Test prediksi
test_texts = [
    'Aplikasi ini sangat bagus untuk belajar bahasa',
    'Tidak berguna, sering error dan lambat',
    'Biasa saja, lumayan',
    'Saya suka sekali aplikasi duolingo ini mantap',
    'Kecewa, tidak bisa login dan sering crash',
    'Aplikasi pembelajaran bahasa yang cukup oke'
]

print('=' * 60)
print('      PREDIKSI SENTIMEN MANUAL')
print('=' * 60)
for text in test_texts:
    label, proba = predict_sentiment(text)
    icon = '🟢' if label == 'positif' else ('🔴' if label == 'negatif' else '🟡')
    print(f'\n📝 Teks    : {text}')
    print(f'{icon} Sentimen: {label.upper()}')
    print(f'   Probabilitas: {proba}')
    print('-' * 60)

## 13. Perbandingan Label Asli vs Prediksi

In [ ]:
# Perbandingan distribusi label asli vs prediksi
fig, ax = plt.subplots(figsize=(10, 6))

labels_name = ['positif', 'negatif', 'netral']
asli_counts = [sum(y_test == l) for l in labels_name]
pred_counts = [sum(y_pred == l) for l in labels_name]

x = np.arange(len(labels_name))
width = 0.35

bars1 = ax.bar(x - width/2, asli_counts, width, label='Label Asli', color='#0984e3', edgecolor='white')
bars2 = ax.bar(x + width/2, pred_counts, width, label='Label Prediksi', color='#e17055', edgecolor='white')

ax.set_xlabel('Label Sentimen', fontsize=12)
ax.set_ylabel('Jumlah', fontsize=12)
ax.set_title('Perbandingan Label Asli vs Label Prediksi', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([l.capitalize() for l in labels_name], fontsize=12)
ax.legend(fontsize=12)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            str(int(bar.get_height())), ha='center', fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            str(int(bar.get_height())), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 14. Kesimpulan

### Ringkasan Hasil Analisis Sentimen Ulasan Duolingo:

1. **Dataset**: Ulasan aplikasi Duolingo dari Google Play Store
2. **Preprocessing**: Case Folding → Cleaning → Tokenizing → Stopword Removal → Stemming (Sastrawi)
3. **Feature Extraction**: TF-IDF (Term Frequency - Inverse Document Frequency)
4. **Pembagian Data**: 80% Training, 20% Testing
5. **Model**: Multinomial Naive Bayes dengan Laplace Smoothing
6. **Klasifikasi**: 3 Kelas (Positif, Negatif, Netral)
7. **Evaluasi**: Accuracy, Precision, Recall, F1-Score

---
**By Ahmad Saifulla** | DuoSentimen Project